# Recovery Reason Analysis

## Objective

Analyze recovery reasons to identify operational bottlenecks affecting claim collection.

This analysis helps management understand why recoveries are delayed, unsuccessful, or transferred to legal processes.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from src.preprocessing import prepare_data

In [126]:
df = prepare_data()

c:\Users\Subhan\OneDrive\Desktop\Road Towards Goals\Reownlogics\Day-1\Insurance_Claims_Analysisdata\notebooks\src\preprocessing.py:80: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [129]:
reason_summary = (
    df.groupby("Recovery Reason")
      .agg(
          Claims=("Claim ID", "count"),
          Recovery=("Recovery Amount", "sum"),
          Collected=("Collected Amount", "sum"),
          Remaining=("Remaining Amount", "sum")
      )
      .reset_index()
)

reason_summary["Collection Rate"] = (
    reason_summary["Collected"]
    / reason_summary["Recovery"]
    * 100
).round(2)

reason_summary["Avg Recovery"] = (
    reason_summary["Recovery"]
    / reason_summary["Claims"]
).round(2)

reason_summary = reason_summary.sort_values(
    "Claims",
    ascending=False
)

reason_summary.head(15)

,Recovery Reason,Claims,Recovery,Collected,Remaining,Collection Rate,Avg Recovery
21,لا يملك رخصة قيادة,1532,18088633.71,549684.23,17538938.04,3.04,11807.20
0,أنتهاء تاريخ رخصة القيادة,579,3832485.57,144175.36,3688309.36,3.76,6619.15
16,عكس اتجاه السير,265,3663126.98,196327.92,3466798.36,5.36,13823.12
11,تجاوز الإشارة الحمراء,126,2196916.76,107245.02,2089670.94,4.88,17435.85
14,صدم و هروب من موقع الحادث,118,1948312.83,14313.10,1933999.73,0.73,16511.13
20,لا يملك تأمين ساري,96,654903.91,33539.65,621364.26,5.12,6821.92
13,حالات حق الرجوع / الافراد,90,1059649.01,34678.35,1024970.66,3.27,11773.88
1,اخفاء حقيقة جوهرية / تبديل السائق وقت الحادث,87,397187.86,87707.29,309480.57,22.08,4565.38
27,نوع الرخصة لا يخوله بقيادة المركبة,33,571105.10,3899.75,567205.00,0.68,17306.22
18,عمر السائق أقل من 18 عام وغير مذكور في الوثيقة,6,103796.02,0.00,103796.02,0.00,17299.34


### Top Recovery Reasons

In [140]:
fig = px.bar(
    reason_summary.head(15),
    x="Claims",
    y="Recovery Reason",
    orientation="h",
    color="Claims",
    text="Claims",
    title="Top Recovery Reasons by Claim Volume"
)

fig.show()

### Recovery Amount by Reason

In [131]:
fig = px.bar(
    reason_summary.sort_values("Recovery").tail(15),
    x="Recovery",
    y="Recovery Reason",
    orientation="h",
    color="Recovery",
    text_auto=".2s",
    title="Recovery Amount by Recovery Reason"
)

fig.show()

### Remaining Amount by Reason

In [132]:
fig = px.bar(
    reason_summary.sort_values("Remaining").tail(15),
    x="Remaining",
    y="Recovery Reason",
    orientation="h",
    color="Remaining",
    text_auto=".2s",
    title="Outstanding Recovery by Reason"
)

fig.show()

### Collection Rate by Reason

In [133]:
reason_filtered = (
    reason_summary[
        reason_summary["Claims"] >= 10
    ]
)

fig = px.bar(
    reason_filtered.sort_values("Collection Rate"),
    x="Collection Rate",
    y="Recovery Reason",
    orientation="h",
    color="Collection Rate",
    text="Collection Rate",
    title="Collection Rate by Recovery Reason"
)

fig.show()

### Bubble Chart

In [134]:
fig = px.scatter(
    reason_filtered,
    x="Recovery",
    y="Collection Rate",
    size="Claims",
    color="Remaining",
    hover_name="Recovery Reason",
    title="Recovery Reason Performance Matrix"
)

fig.show()

### Executive Insights

In [135]:
highest_claims = reason_summary.iloc[0]

highest_remaining = (
    reason_summary
    .sort_values("Remaining", ascending=False)
    .iloc[0]
)

best_collection = (
    reason_filtered
    .sort_values("Collection Rate", ascending=False)
    .iloc[0]
)

print(f"""
📊 RECOVERY REASON INSIGHTS

• Most Common Recovery Reason:
  {highest_claims['Recovery Reason']}
  ({highest_claims['Claims']} claims)

• Largest Outstanding Balance:
  {highest_remaining['Recovery Reason']}
  SAR {highest_remaining['Remaining']:,.0f}

• Highest Collection Rate:
  {best_collection['Recovery Reason']}
  {best_collection['Collection Rate']:.2f}%
""")


📊 RECOVERY REASON INSIGHTS

• Most Common Recovery Reason:
  لا يملك رخصة قيادة
  (1532 claims)

• Largest Outstanding Balance:
  لا يملك رخصة قيادة
  SAR 17,538,938

• Highest Collection Rate:
  اخفاء حقيقة جوهرية / تبديل السائق وقت الحادث
  22.08%



In [136]:
priority_reasons = (
    reason_filtered[
        (reason_filtered["Recovery"] > reason_filtered["Recovery"].median()) &
        (reason_filtered["Collection Rate"] < 20)
    ]
    .sort_values("Remaining", ascending=False)
)

priority_reasons[
    [
        "Recovery Reason",
        "Recovery",
        "Remaining",
        "Collection Rate"
    ]
]

,Recovery Reason,Recovery,Remaining,Collection Rate
21,لا يملك رخصة قيادة,18088633.71,17538938.04,3.04
0,أنتهاء تاريخ رخصة القيادة,3832485.57,3688309.36,3.76
16,عكس اتجاه السير,3663126.98,3466798.36,5.36
11,تجاوز الإشارة الحمراء,2196916.76,2089670.94,4.88


# Conclusion

This analysis identifies the recovery reasons contributing most to financial exposure and poor collection performance.

Management should prioritize reasons associated with large outstanding balances and consistently low collection rates to improve overall recovery efficiency.